# Fast Accumulation Offset Explorer: Laser 3

This notebook is for quickly tuning `accumulation_start_offset_us` on `runs/camera/laser_selected_cycle_40px_dot123_test1_laser_3-sync-check`. It avoids the slow 30-subplot matplotlib redraw by rendering one cached PNG contact sheet with ROI zoom, optional event dilation, and per-frame contrast.

Trigger order is preserved. For this run, 10 cycles x 3 numbers = 30 ordered frames. Display labels default to the observed no-bitplane-remap order `2,3,1`; change `display_sequence` if the hardware order changes.

If the run metadata contains `startup_leader.trigger_count`, those blank startup-leader triggers are skipped before event-range alignment and cycle selection. They are real TRIG_OUT_2 pulses, but they are not numbered display frames.



In [ ]:
from __future__ import annotations

from functools import lru_cache
from io import BytesIO
from math import ceil
from pathlib import Path
import json
import sys
import time

import numpy as np
try:
    from IPython.display import Image as DisplayImage, clear_output, display
except ModuleNotFoundError:
    class DisplayImage:
        def __init__(self, data=None, **kwargs):
            self.data = data

    def clear_output(*args, **kwargs):
        return None

    def display(obj):
        print(type(obj).__name__)
from PIL import Image, ImageDraw, ImageFont

try:
    import cv2
except ModuleNotFoundError:
    cv2 = None


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "dmdcontrol").is_dir() and (candidate / "runs").is_dir():
            return candidate
    raise RuntimeError("Could not find repository root from the current notebook directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUN_DIR = REPO_ROOT / "runs" / "camera" / "laser_selected_cycle_40px_dot123_test1_laser_3-sync-check"
AEDAT4_PATH = RUN_DIR / "raw.aedat4"
METADATA_PATH = RUN_DIR / "metadata.json"
SUMMARY_PATH = RUN_DIR / "summary.json"

print(f"run dir: {RUN_DIR}")
print(f"aedat4 exists: {AEDAT4_PATH.exists()}")

In [ ]:
from dmdcontrol.camera.accumulation import accumulate_events_for_triggers
from dmdcontrol.camera.reprocess_aedat4 import read_aedat4_recording
from dmdcontrol.camera.runs import _events_to_arrays, _process_accumulation_triggers, _trigger_timestamps


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def _startup_leader_trigger_count(metadata: dict, artifact_summary: dict) -> int:
    # Paired capture now displays blank startup-leader frames after the DLPC sequencers start.
    # Their TRIG_OUT_2 pulses are real, but they are not semantic display frames, so all
    # analysis must skip them before assigning labels to trigger windows.
    leader = metadata.get("startup_leader") or {}
    if leader.get("trigger_count") is not None:
        return int(leader.get("trigger_count") or 0)
    leader_skip = artifact_summary.get("startup_leader_skip") or {}
    return int(
        leader_skip.get("requested_trigger_count")
        if leader_skip.get("requested_trigger_count") is not None
        else leader_skip.get("skipped_trigger_count") or 0
    )


metadata = load_json(METADATA_PATH)
summary = load_json(SUMMARY_PATH)
artifact_summary = metadata.get("artifact_summary") or summary
number_sequence = list(metadata.get("number_sequence") or [])
cycle_length = len(number_sequence) or int(metadata.get("expected_trigger_count") or 5)
default_cycles = int(metadata.get("accumulation_cycles") or 10)
default_window_us = int(
    metadata.get("accumulation_window_us")
    or metadata.get("exposure_us")
    or artifact_summary.get("window_us")
    or 1500
)
metadata_offset_us = metadata.get("accumulation_start_offset_us")
summary_offset_us = artifact_summary.get("window_start_offset_us")
default_dark_time_us = int(metadata.get("dark_time_us") or 1000)
default_offset_us = -int(default_window_us + default_dark_time_us)
trigger_period_us = int(default_window_us + default_dark_time_us)
offset_min_us = -int(trigger_period_us + 1000)
offset_max_us = int(trigger_period_us + 1000)
startup_leader_trigger_count = _startup_leader_trigger_count(metadata, artifact_summary)
# This command did not use --numbers-bitplane-order. From the earlier observed DMD chronology,
# the trigger order for [1,2,3] displays as [2,3,1]. Frames are not reordered; only labels change.
display_sequence = [2, 3, 1] if number_sequence == [1, 2, 3] and metadata.get("numbers_bitplane_order") is None else list(number_sequence)

t0 = time.perf_counter()
recording = read_aedat4_recording(AEDAT4_PATH)
event_arrays = _events_to_arrays(recording.events)
event_timestamps = event_arrays["t"]
FORCE_AEDAT4_TRIGGERS = True
trigger_source = "raw AEDAT4 rising triggers"
default_trigger_stages = _process_accumulation_triggers(
    recording.triggers,
    event_timestamps,
    window_us=default_window_us,
    window_start_offset_us=default_offset_us,
    max_accumulation_triggers=None,
    trigger_cycle_length=cycle_length,
    accumulation_cycles=None,
    startup_leader_trigger_count=startup_leader_trigger_count,
)
available_cycles = int(default_trigger_stages.cycle_limit_metadata.get("available_full_cycles") or 0)
default_cycles = max(1, min(default_cycles, max(1, available_cycles)))
load_s = time.perf_counter() - t0

width, height = recording.resolution
print(f"loaded AEDAT4 in {load_s:.2f}s")
print(f"resolution: {width} x {height}")
print(f"events: {recording.stats['event_count']:,}; triggers: {recording.stats['trigger_count']:,}")
print(f"trigger edges: {recording.stats['trigger_edges']}")
print(f"trigger source for accumulation: {trigger_source}; available cycles after leader/alignment: {available_cycles}")
print(f"startup leader trigger skip: {startup_leader_trigger_count}")
print(f"cycle length: {cycle_length}; default cycles: {default_cycles}; default window: {default_window_us} us")
print(f"default offset: {default_offset_us} us (pre-trigger exposure window); offset slider range: {offset_min_us}..{offset_max_us} us")
print(f"metadata offset: {metadata_offset_us}; summary offset: {summary_offset_us}")
print(f"metadata number sequence: {number_sequence}; display labels: {display_sequence}")


In [ ]:
@lru_cache(maxsize=8)
def accumulate_cached(offset_us: int, cycles: int, window_us: int, polarity_mode: str):
    requested_cycles = int(cycles)
    stages = _process_accumulation_triggers(
        recording.triggers,
        event_timestamps,
        window_us=int(window_us),
        window_start_offset_us=int(offset_us),
        max_accumulation_triggers=None,
        trigger_cycle_length=cycle_length,
        accumulation_cycles=requested_cycles,
        startup_leader_trigger_count=startup_leader_trigger_count,
    )
    selected = stages.final
    alignment = stages.alignment_metadata
    cycle_info = stages.cycle_limit_metadata
    frames = accumulate_events_for_triggers(
        recording.events,
        selected,
        resolution=recording.resolution,
        window_us=int(window_us),
        polarity_mode=str(polarity_mode),
        window_start_offset_us=int(offset_us),
    )
    return frames, _trigger_timestamps(selected), alignment, cycle_info


def frame_values(frames: np.ndarray, view: str, tone: str, gamma: float) -> tuple[np.ndarray, bool]:
    source = frames.astype(np.float32, copy=False)
    if view == "signed":
        return source, True
    if view == "positive":
        values = np.maximum(source, 0)
    elif view == "negative":
        values = np.maximum(-source, 0)
    else:
        values = np.abs(source)
    if tone == "log":
        values = np.log1p(values)
    elif tone == "gamma":
        values = np.power(values, float(gamma))
    return values, False


def auto_roi(frames: np.ndarray, pad: int, flip_x: bool, crop_mode: str) -> tuple[slice, slice]:
    if crop_mode == "full":
        return slice(0, frames.shape[1]), slice(0, frames.shape[2])
    source = frames[:, :, ::-1] if flip_x else frames
    mask = np.any(np.abs(source) > 0, axis=0)
    if not np.any(mask):
        return slice(0, frames.shape[1]), slice(0, frames.shape[2])
    ys, xs = np.where(mask)
    y0 = max(0, int(ys.min()) - int(pad))
    y1 = min(frames.shape[1], int(ys.max()) + int(pad) + 1)
    x0 = max(0, int(xs.min()) - int(pad))
    x1 = min(frames.shape[2], int(xs.max()) + int(pad) + 1)
    if crop_mode == "square":
        side = max(y1 - y0, x1 - x0)
        cy = (y0 + y1) // 2
        cx = (x0 + x1) // 2
        y0 = max(0, min(frames.shape[1] - side, cy - side // 2))
        x0 = max(0, min(frames.shape[2] - side, cx - side // 2))
        y1 = min(frames.shape[1], y0 + side)
        x1 = min(frames.shape[2], x0 + side)
    return slice(y0, y1), slice(x0, x1)


def dilate_plane(plane: np.ndarray, dot_size: int) -> np.ndarray:
    dot_size = int(dot_size)
    if dot_size <= 1:
        return plane
    if cv2 is not None:
        kernel = np.ones((dot_size, dot_size), dtype=np.uint8)
        return cv2.dilate(plane.astype(np.float32, copy=False), kernel)
    padded = np.pad(plane, dot_size // 2, mode="edge")
    out = np.zeros_like(plane)
    for dy in range(dot_size):
        for dx in range(dot_size):
            out = np.maximum(out, padded[dy:dy + plane.shape[0], dx:dx + plane.shape[1]])
    return out


def scale_limit(values: np.ndarray, signed: bool, percentile: float) -> float:
    source = np.abs(values) if signed else values
    nonzero = source[source > 0]
    if nonzero.size == 0:
        return 1.0
    return max(float(np.percentile(nonzero, float(percentile))), 1e-6)


def frame_rgb(values: np.ndarray, signed: bool, limit: float, dot_size: int) -> np.ndarray:
    if signed:
        pos = dilate_plane(np.maximum(values, 0), dot_size)
        neg = dilate_plane(np.maximum(-values, 0), dot_size)
        pos8 = np.clip(pos / limit * 255, 0, 255).astype(np.uint8)
        neg8 = np.clip(neg / limit * 255, 0, 255).astype(np.uint8)
        rgb = np.zeros((*values.shape, 3), dtype=np.uint8)
        rgb[..., 0] = pos8
        rgb[..., 1] = neg8
        rgb[..., 2] = neg8
        return rgb
    plane = dilate_plane(values, dot_size)
    gray = np.clip(plane / limit * 255, 0, 255).astype(np.uint8)
    return np.repeat(gray[..., None], 3, axis=2)


def resize_nearest(rgb: np.ndarray, scale: int) -> Image.Image:
    image = Image.fromarray(rgb, mode="RGB")
    if int(scale) <= 1:
        return image
    return image.resize((image.width * int(scale), image.height * int(scale)), Image.Resampling.NEAREST)


def png_bytes(image: Image.Image) -> bytes:
    buffer = BytesIO()
    image.save(buffer, format="PNG", optimize=False)
    return buffer.getvalue()

In [ ]:
FONT = ImageFont.load_default()


def render_fast_sheet(
    offset_us: int = default_offset_us,
    cycles: int = default_cycles,
    window_us: int = default_window_us,
    polarity_mode: str = "ignore",
    view: str = "magnitude",
    tone: str = "log",
    gamma: float = 0.65,
    contrast: str = "per-frame",
    vmax_percentile: float = 99.0,
    crop_mode: str = "square",
    crop_pad: int = 28,
    dot_size: int = 2,
    tile_scale: int = 2,
    focus_frame: int = 1,
    focus_scale: int = 4,
    flip_x: bool = True,
):
    start = time.perf_counter()
    frames, trigger_ts, alignment, cycle_info = accumulate_cached(
        int(offset_us), int(cycles), int(window_us), str(polarity_mode)
    )
    frames_for_view = frames[:, :, ::-1] if flip_x else frames
    y_slice, x_slice = auto_roi(frames, crop_pad, flip_x=flip_x, crop_mode=crop_mode)
    cropped = frames_for_view[:, y_slice, x_slice]
    values, signed = frame_values(cropped, view=view, tone=tone, gamma=gamma)
    frame_count = values.shape[0]
    cols = max(1, int(cycle_length))
    rows = max(1, ceil(frame_count / cols))

    if contrast == "global":
        global_limit = scale_limit(values, signed=signed, percentile=vmax_percentile)
    else:
        global_limit = None

    tile_images = []
    for index in range(frame_count):
        limit = global_limit or scale_limit(values[index], signed=signed, percentile=vmax_percentile)
        tile_images.append(resize_nearest(frame_rgb(values[index], signed, limit, dot_size), tile_scale))

    tile_w = tile_images[0].width if tile_images else 1
    tile_h = tile_images[0].height if tile_images else 1
    label_h = 26
    header_h = 58
    gap = 8
    focus_index = max(0, min(frame_count - 1, int(focus_frame) - 1)) if frame_count else 0
    focus_limit = global_limit or scale_limit(values[focus_index], signed=signed, percentile=vmax_percentile)
    focus_image = resize_nearest(frame_rgb(values[focus_index], signed, focus_limit, max(1, int(dot_size))), focus_scale)

    sheet_w = cols * tile_w + (cols - 1) * gap
    sheet_h = rows * (tile_h + label_h) + (rows - 1) * gap
    canvas_w = max(sheet_w, focus_image.width) + 20
    canvas_h = header_h + focus_image.height + 22 + sheet_h + 24
    canvas = Image.new("RGB", (canvas_w, canvas_h), (14, 14, 14))
    draw = ImageDraw.Draw(canvas)

    counts = np.count_nonzero(np.abs(frames) > 0, axis=(1, 2)) if frame_count else np.array([], dtype=int)
    roi_text = f"roi y={y_slice.start}:{y_slice.stop} x={x_slice.start}:{x_slice.stop}"
    title = (
        f"offset {int(offset_us)} us | window {int(window_us)} us | {frame_count} frames | "
        f"{polarity_mode}/{view}/{tone} | contrast={contrast} | dot={int(dot_size)}"
    )
    draw.text((10, 8), title, fill=(235, 235, 235), font=FONT)
    draw.text((10, 28), roi_text, fill=(180, 180, 180), font=FONT)
    if counts.size:
        draw.text(
            (10, 44),
            f"nonzero pixels: first={int(counts[0])}, min={int(counts.min())}, median={int(np.median(counts))}, max={int(counts.max())}",
            fill=(180, 180, 180),
            font=FONT,
        )

    focus_x = (canvas_w - focus_image.width) // 2
    focus_y = header_h
    canvas.paste(focus_image, (focus_x, focus_y))
    focus_slot = focus_index % cols
    focus_cycle = focus_index // cols + 1
    focus_label = display_sequence[focus_slot] if focus_slot < len(display_sequence) else focus_slot + 1
    draw.text(
        (10, focus_y + focus_image.height + 4),
        f"focus frame {focus_index + 1}: expected {focus_label}, cycle {focus_cycle}, nonzero={int(counts[focus_index]) if counts.size else 0}",
        fill=(235, 235, 235),
        font=FONT,
    )

    first_trigger = int(trigger_ts[0]) if len(trigger_ts) else 0
    grid_y = focus_y + focus_image.height + 26
    for index, image in enumerate(tile_images):
        row = index // cols
        col = index % cols
        x = 10 + col * (tile_w + gap)
        y = grid_y + row * (tile_h + label_h + gap)
        canvas.paste(image, (x, y + label_h))
        slot = index % cols
        cycle = index // cols + 1
        label = display_sequence[slot] if slot < len(display_sequence) else slot + 1
        dt = int(trigger_ts[index] - first_trigger) if len(trigger_ts) else 0
        color = (255, 245, 170) if index == focus_index else (220, 220, 220)
        draw.text((x, y), f"{index + 1}: {label} c{cycle} +{dt}us", fill=color, font=FONT)
        if counts.size:
            draw.text((x, y + 12), f"nz={int(counts[index])}", fill=(170, 170, 170), font=FONT)

    elapsed_ms = (time.perf_counter() - start) * 1000
    draw.text((canvas_w - 115, 8), f"{elapsed_ms:.0f} ms", fill=(120, 210, 120), font=FONT)
    return canvas, frames, counts, alignment, cycle_info


image, frames, counts, alignment, cycle_info = render_fast_sheet(offset_us=default_offset_us)
display(DisplayImage(data=png_bytes(image)))

## Trigger Timeline and Bleed Diagnostics

These views answer different questions than the frame grid. The timeline shows trigger positions, accumulation windows, and event density over time. The relative activity graph shows whether events are arriving before, during, or after each trigger. The bleed matrix estimates how much spatial activity carries from one ordered frame into the next.

In [ ]:
SLOT_COLORS = [
    (95, 190, 255),
    (255, 175, 80),
    (140, 220, 120),
    (235, 105, 130),
    (190, 150, 255),
]


def slot_color(slot: int) -> tuple[int, int, int]:
    return SLOT_COLORS[int(slot) % len(SLOT_COLORS)]


def render_trigger_timeline(
    offset_us: int = default_offset_us,
    cycles: int = default_cycles,
    window_us: int = default_window_us,
    bin_us: int = 100,
    margin_us: int = 500,
    polarity_mode: str = "ignore",
):
    frames, trigger_ts, alignment, cycle_info = accumulate_cached(
        int(offset_us), int(cycles), int(window_us), str(polarity_mode)
    )
    trigger_ts = np.asarray(trigger_ts, dtype=np.int64)
    if trigger_ts.size == 0:
        return Image.new("RGB", (900, 120), (14, 14, 14))

    # Keep the absolute-time axis anchored to triggers, not to shifted windows.
    # Otherwise changing offset re-centers the graph and makes trigger lines look like they move backward.
    axis_offset_min_us = int(offset_min_us)
    axis_offset_max_us = int(offset_max_us)
    start = int(np.min(trigger_ts) + min(axis_offset_min_us, int(offset_us), 0) - int(margin_us))
    end = int(np.max(trigger_ts) + max(axis_offset_max_us, int(offset_us), 0) + int(window_us) + int(margin_us))
    bins = np.arange(start, end + int(bin_us), int(bin_us), dtype=np.int64)
    if bins.size < 2:
        bins = np.array([start, end + 1], dtype=np.int64)
    event_counts, _ = np.histogram(event_timestamps, bins=bins)

    plot_w = 1160
    left = 58
    right = 16
    top_h = 128
    row_h = 30
    footer_h = 42
    rows = max(1, ceil(len(trigger_ts) / cycle_length))
    canvas = Image.new("RGB", (plot_w + left + right, top_h + rows * row_h + footer_h), (14, 14, 14))
    draw = ImageDraw.Draw(canvas)

    def x_of(t_us: int | float) -> int:
        return int(left + (float(t_us) - start) / max(1, end - start) * plot_w)

    draw.text((10, 8), f"trigger/window timeline | offset {int(offset_us)} us | window {int(window_us)} us | bin {int(bin_us)} us", fill=(235, 235, 235), font=FONT)
    draw.text((10, 26), "top graph = all event timestamps binned over time; rows = ordered cycles", fill=(175, 175, 175), font=FONT)

    hist_top = 54
    hist_h = 60
    max_count = max(1, int(event_counts.max()) if event_counts.size else 1)
    for idx, count in enumerate(event_counts):
        x0 = x_of(int(bins[idx]))
        x1 = max(x0 + 1, x_of(int(bins[idx + 1])))
        h = int((int(count) / max_count) * hist_h)
        draw.rectangle((x0, hist_top + hist_h - h, x1, hist_top + hist_h), fill=(95, 95, 95))
    draw.rectangle((left, hist_top, left + plot_w, hist_top + hist_h), outline=(90, 90, 90))

    for index, trigger in enumerate(trigger_ts):
        slot = index % cycle_length
        color = slot_color(slot)
        x = x_of(int(trigger))
        draw.line((x, hist_top, x, hist_top + hist_h), fill=color)

    for cycle in range(rows):
        y = top_h + cycle * row_h
        draw.text((10, y + 7), f"c{cycle + 1}", fill=(210, 210, 210), font=FONT)
        draw.line((left, y + row_h // 2, left + plot_w, y + row_h // 2), fill=(45, 45, 45))
        for slot in range(cycle_length):
            index = cycle * cycle_length + slot
            if index >= len(trigger_ts):
                break
            trigger = int(trigger_ts[index])
            win_start = trigger + int(offset_us)
            win_end = win_start + int(window_us)
            x0 = x_of(win_start)
            x1 = x_of(win_end)
            xt = x_of(trigger)
            color = slot_color(slot)
            fill = tuple(max(25, c // 3) for c in color)
            draw.rectangle((x0, y + 5, x1, y + row_h - 6), fill=fill, outline=color)
            draw.line((xt, y + 2, xt, y + row_h - 2), fill=(255, 255, 255))
            if index + 1 < len(trigger_ts) and win_end > int(trigger_ts[index + 1]):
                draw.line((x1, y + 3, x1, y + row_h - 3), fill=(255, 80, 80), width=2)

    dt = np.diff(trigger_ts)
    footer_y = top_h + rows * row_h + 8
    if dt.size:
        min_gap_after_window = int(np.min(dt - int(window_us) - int(offset_us)))
        draw.text(
            (10, footer_y),
            f"trigger spacing us: min={int(dt.min())}, median={float(np.median(dt)):.0f}, max={int(dt.max())}; min next-trigger gap after window={min_gap_after_window} us",
            fill=(210, 210, 210),
            font=FONT,
        )
    draw.text((10, footer_y + 16), "colored blocks = accumulation windows; white line = trigger; red marker = window overlaps next trigger", fill=(175, 175, 175), font=FONT)
    return canvas


def render_relative_activity(
    offset_us: int = default_offset_us,
    cycles: int = default_cycles,
    window_us: int = default_window_us,
    pre_us: int = 1000,
    post_us: int = max(3000, default_window_us + 1000),
    bin_us: int = 50,
    polarity_mode: str = "ignore",
):
    frames, trigger_ts, alignment, cycle_info = accumulate_cached(
        int(offset_us), int(cycles), int(window_us), str(polarity_mode)
    )
    trigger_ts = np.asarray(trigger_ts, dtype=np.int64)
    rel_edges = np.arange(-int(pre_us), int(post_us) + int(bin_us), int(bin_us), dtype=np.int64)
    rel_centers = (rel_edges[:-1] + rel_edges[1:]) / 2.0
    slot_counts = np.zeros((cycle_length, len(rel_centers)), dtype=np.float32)
    slot_seen = np.zeros(cycle_length, dtype=np.int64)
    for index, trigger in enumerate(trigger_ts):
        slot = index % cycle_length
        hist, _ = np.histogram(event_timestamps, bins=trigger + rel_edges)
        slot_counts[slot] += hist.astype(np.float32)
        slot_seen[slot] += 1
    for slot in range(cycle_length):
        if slot_seen[slot] > 0:
            slot_counts[slot] /= slot_seen[slot]

    W, H = 1000, 360
    left, right, top, bottom = 68, 18, 40, 52
    plot_w = W - left - right
    plot_h = H - top - bottom
    canvas = Image.new("RGB", (W, H), (14, 14, 14))
    draw = ImageDraw.Draw(canvas)

    def x_of(rel_us: int | float) -> int:
        return int(left + (float(rel_us) + int(pre_us)) / max(1, int(pre_us) + int(post_us)) * plot_w)

    max_y = max(1.0, float(slot_counts.max()))

    def y_of(count: int | float) -> int:
        return int(top + plot_h - (float(count) / max_y) * plot_h)

    win_x0 = x_of(int(offset_us))
    win_x1 = x_of(int(offset_us) + int(window_us))
    draw.rectangle((win_x0, top, win_x1, top + plot_h), fill=(32, 42, 32), outline=(90, 130, 90))
    draw.line((x_of(0), top, x_of(0), top + plot_h), fill=(230, 230, 230))
    draw.rectangle((left, top, left + plot_w, top + plot_h), outline=(95, 95, 95))
    draw.text((10, 8), f"event activity relative to trigger | offset {int(offset_us)} us | window {int(window_us)} us", fill=(235, 235, 235), font=FONT)
    draw.text((left, H - 34), "relative time from trigger (us)", fill=(190, 190, 190), font=FONT)
    draw.text((8, top + 4), "avg events/bin", fill=(190, 190, 190), font=FONT)
    for tick in range(-int(pre_us), int(post_us) + 1, 500):
        x = x_of(tick)
        draw.line((x, top + plot_h, x, top + plot_h + 5), fill=(140, 140, 140))
        draw.text((x - 16, top + plot_h + 8), str(tick), fill=(160, 160, 160), font=FONT)

    for slot in range(cycle_length):
        points = [(x_of(x), y_of(y)) for x, y in zip(rel_centers, slot_counts[slot], strict=False)]
        if len(points) >= 2:
            draw.line(points, fill=slot_color(slot), width=2)
        label = display_sequence[slot] if slot < len(display_sequence) else slot + 1
        draw.text((W - 138, 44 + slot * 16), f"{label}: avg peak {float(slot_counts[slot].max()):.1f}", fill=slot_color(slot), font=FONT)
    draw.text((win_x0 + 3, top + 4), "accum window", fill=(170, 220, 170), font=FONT)
    return canvas


def render_bleed_matrix(
    offset_us: int = default_offset_us,
    cycles: int = default_cycles,
    window_us: int = default_window_us,
    polarity_mode: str = "ignore",
    mask_dilate: int = 3,
):
    frames, trigger_ts, alignment, cycle_info = accumulate_cached(
        int(offset_us), int(cycles), int(window_us), str(polarity_mode)
    )
    masks = np.abs(frames) > 0
    if int(mask_dilate) > 1:
        masks = np.stack([dilate_plane(mask.astype(np.float32), int(mask_dilate)) > 0 for mask in masks], axis=0)
    active = masks.reshape(masks.shape[0], -1).sum(axis=1) if masks.size else np.array([], dtype=np.int64)
    matrix_values = [[[] for _ in range(cycle_length)] for _ in range(cycle_length)]
    adjacent_scores = []
    for index in range(max(0, len(masks) - 1)):
        a = index % cycle_length
        b = (index + 1) % cycle_length
        inter = int(np.logical_and(masks[index], masks[index + 1]).sum())
        denom = max(1, int(min(active[index], active[index + 1])))
        score = inter / denom
        matrix_values[a][b].append(score)
        adjacent_scores.append(score)

    matrix = np.zeros((cycle_length, cycle_length), dtype=np.float32)
    for r in range(cycle_length):
        for c in range(cycle_length):
            if matrix_values[r][c]:
                matrix[r, c] = float(np.mean(matrix_values[r][c]))

    cell = 74
    left, top = 116, 72
    W = left + cycle_length * cell + 260
    H = top + cycle_length * cell + 86
    canvas = Image.new("RGB", (W, H), (14, 14, 14))
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 8), f"consecutive-frame spatial overlap | offset {int(offset_us)} us | mask dilate {int(mask_dilate)}", fill=(235, 235, 235), font=FONT)
    draw.text((10, 27), "cell value = mean intersection / smaller active-pixel count for ordered frame transitions", fill=(175, 175, 175), font=FONT)
    draw.text((10, 47), "high off-diagonal values can mean bleed, but shared strokes in the same position also contribute", fill=(175, 175, 175), font=FONT)

    for slot in range(cycle_length):
        label = str(display_sequence[slot] if slot < len(display_sequence) else slot + 1)
        draw.text((left + slot * cell + 28, top - 22), label, fill=slot_color(slot), font=FONT)
        draw.text((left - 34, top + slot * cell + 28), label, fill=slot_color(slot), font=FONT)
    draw.text((left + 95, top - 42), "next frame", fill=(210, 210, 210), font=FONT)
    draw.text((22, top + 145), "current", fill=(210, 210, 210), font=FONT)

    max_value = max(0.01, float(matrix.max()))
    for r in range(cycle_length):
        for c in range(cycle_length):
            value = float(matrix[r, c])
            intensity = int(np.clip(value / max_value * 255, 0, 255))
            fill = (intensity, max(18, intensity // 3), 28)
            x0 = left + c * cell
            y0 = top + r * cell
            draw.rectangle((x0, y0, x0 + cell - 4, y0 + cell - 4), fill=fill, outline=(85, 85, 85))
            draw.text((x0 + 13, y0 + 27), f"{value * 100:.1f}%", fill=(245, 245, 245), font=FONT)

    stats_x = left + cycle_length * cell + 32
    if active.size:
        draw.text((stats_x, top), "active px/frame", fill=(230, 230, 230), font=FONT)
        draw.text((stats_x, top + 18), f"first: {int(active[0])}", fill=(190, 190, 190), font=FONT)
        draw.text((stats_x, top + 34), f"min:   {int(active.min())}", fill=(190, 190, 190), font=FONT)
        draw.text((stats_x, top + 50), f"med:   {int(np.median(active))}", fill=(190, 190, 190), font=FONT)
        draw.text((stats_x, top + 66), f"max:   {int(active.max())}", fill=(190, 190, 190), font=FONT)
    if adjacent_scores:
        draw.text((stats_x, top + 102), "adjacent overlap", fill=(230, 230, 230), font=FONT)
        draw.text((stats_x, top + 120), f"mean: {float(np.mean(adjacent_scores)) * 100:.1f}%", fill=(190, 190, 190), font=FONT)
        draw.text((stats_x, top + 136), f"max:  {float(np.max(adjacent_scores)) * 100:.1f}%", fill=(190, 190, 190), font=FONT)
    return canvas


timeline_image = render_trigger_timeline(offset_us=default_offset_us)
activity_image = render_relative_activity(offset_us=default_offset_us)
bleed_image = render_bleed_matrix(offset_us=default_offset_us)
display(DisplayImage(data=png_bytes(timeline_image)))
display(DisplayImage(data=png_bytes(activity_image)))
display(DisplayImage(data=png_bytes(bleed_image)))

## Interactive Controls

Defaults are tuned to make sparse event frames visible: ROI crop, per-frame contrast, log tone, and dot dilation. Switch contrast to `global` when you want brightness comparisons to be physically meaningful across frames.

In [ ]:
try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ModuleNotFoundError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed in this kernel. Run `%pip install ipywidgets`, restart the kernel, then rerun this notebook.")


ENABLE_INTERACTIVE_WIDGETS = bool(globals().get("ENABLE_INTERACTIVE_WIDGETS", True))

if HAVE_WIDGETS and ENABLE_INTERACTIVE_WIDGETS:
    offset_slider = widgets.IntSlider(
        value=default_offset_us,
        min=offset_min_us,
        max=offset_max_us,
        step=25,
        description="offset us",
        continuous_update=True,
        layout=widgets.Layout(width="560px"),
    )
    cycles_slider = widgets.IntSlider(
        value=default_cycles,
        min=1,
        max=max(1, min(20, int(available_cycles))),
        step=1,
        description="cycles",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    window_slider = widgets.IntSlider(
        value=default_window_us,
        min=100,
        max=max(2500, default_window_us * 2),
        step=25,
        description="window us",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    focus_slider = widgets.IntSlider(
        value=1,
        min=1,
        max=max(1, default_cycles * cycle_length),
        step=1,
        description="focus",
        continuous_update=True,
        layout=widgets.Layout(width="560px"),
    )
    polarity_dropdown = widgets.Dropdown(value="ignore", options=["ignore", "signed", "positive"], description="accum")
    view_dropdown = widgets.Dropdown(value="magnitude", options=["magnitude", "positive", "negative", "signed"], description="view")
    tone_dropdown = widgets.Dropdown(value="log", options=["log", "gamma", "linear"], description="tone")
    contrast_dropdown = widgets.Dropdown(value="per-frame", options=["per-frame", "global"], description="contrast")
    crop_dropdown = widgets.Dropdown(value="square", options=["square", "auto", "full"], description="crop")
    gamma_slider = widgets.FloatSlider(value=0.65, min=0.2, max=1.5, step=0.05, description="gamma", continuous_update=False, layout=widgets.Layout(width="560px"))
    vmax_slider = widgets.FloatSlider(value=99.0, min=90.0, max=100.0, step=0.1, description="vmax %", continuous_update=False, layout=widgets.Layout(width="560px"))
    pad_slider = widgets.IntSlider(value=28, min=0, max=80, step=2, description="crop pad", continuous_update=False, layout=widgets.Layout(width="560px"))
    dot_slider = widgets.IntSlider(value=2, min=1, max=6, step=1, description="dot size", continuous_update=True, layout=widgets.Layout(width="560px"))
    tile_scale_slider = widgets.IntSlider(value=2, min=1, max=4, step=1, description="tile scale", continuous_update=False, layout=widgets.Layout(width="560px"))
    focus_scale_slider = widgets.IntSlider(value=4, min=2, max=8, step=1, description="focus scale", continuous_update=False, layout=widgets.Layout(width="560px"))
    flip_checkbox = widgets.Checkbox(value=True, description="flip x for viewing")
    out = widgets.Output()

    def sync_focus_max(*_):
        focus_slider.max = max(1, cycles_slider.value * cycle_length)
        if focus_slider.value > focus_slider.max:
            focus_slider.value = focus_slider.max

    def redraw(_=None):
        sync_focus_max()
        with out:
            clear_output(wait=True)
            image, frames, counts, alignment, cycle_info = render_fast_sheet(
                offset_us=offset_slider.value,
                cycles=cycles_slider.value,
                window_us=window_slider.value,
                polarity_mode=polarity_dropdown.value,
                view=view_dropdown.value,
                tone=tone_dropdown.value,
                gamma=gamma_slider.value,
                contrast=contrast_dropdown.value,
                vmax_percentile=vmax_slider.value,
                crop_mode=crop_dropdown.value,
                crop_pad=pad_slider.value,
                dot_size=dot_slider.value,
                tile_scale=tile_scale_slider.value,
                focus_frame=focus_slider.value,
                focus_scale=focus_scale_slider.value,
                flip_x=flip_checkbox.value,
            )
            display(DisplayImage(data=png_bytes(image)))

    for widget in [
        offset_slider,
        cycles_slider,
        window_slider,
        focus_slider,
        polarity_dropdown,
        view_dropdown,
        tone_dropdown,
        contrast_dropdown,
        crop_dropdown,
        gamma_slider,
        vmax_slider,
        pad_slider,
        dot_slider,
        tile_scale_slider,
        focus_scale_slider,
        flip_checkbox,
    ]:
        widget.observe(redraw, names="value")

    controls = widgets.VBox([
        offset_slider,
        cycles_slider,
        window_slider,
        focus_slider,
        widgets.HBox([polarity_dropdown, view_dropdown, tone_dropdown, contrast_dropdown, crop_dropdown]),
        gamma_slider,
        vmax_slider,
        pad_slider,
        widgets.HBox([dot_slider, tile_scale_slider, focus_scale_slider, flip_checkbox]),
    ])
    display(controls)
    display(out)
    redraw()
else:
    image, frames, counts, alignment, cycle_info = render_fast_sheet(offset_us=default_offset_us)
    display(DisplayImage(data=png_bytes(image)))


## Rainbow Timestamp Provenance View

This view colors raw events by their timestamp within the repeating trigger cycle. The hue gradient repeats for every `1, 2, 3` cycle, so leakage from a neighboring number keeps the neighbor's cycle color when it appears inside another frame's accumulated image.

The raw `display_sequence` remains the trigger/LUT order observed for this run. The rainbow controls add a separate analysis label phase shift so the rendered captions and shape-overlap leakage scores can match the visible digit content without rewriting the underlying trigger order. Tile annotations report pre-trigger, in-window, post-window, previous-phase, current-phase, and next-phase event counts for the selected accumulation window.


In [ ]:
try:
    RAINBOW_FONT = ImageFont.truetype("DejaVuSans.ttf", 24)
except OSError:
    try:
        RAINBOW_FONT = ImageFont.truetype("arial.ttf", 24)
    except OSError:
        RAINBOW_FONT = globals().get("FONT", ImageFont.load_default())

STATIC_RAINBOW_VIEW = globals().get("STATIC_RAINBOW_VIEW", "shape")
DEFAULT_RAINBOW_FRAME_COUNT = max(1, min(30, int(available_cycles) * int(cycle_length)))
STATIC_RAINBOW_FRAME_COUNT = int(globals().get("STATIC_RAINBOW_FRAME_COUNT", DEFAULT_RAINBOW_FRAME_COUNT))
ENABLE_INTERACTIVE_WIDGETS = bool(globals().get("ENABLE_INTERACTIVE_WIDGETS", True))
DEFAULT_RAINBOW_LABEL_PHASE_SHIFT = int(globals().get("RAINBOW_LABEL_PHASE_SHIFT", 1 if list(display_sequence) == [2, 3, 1] else 0))
DEFAULT_RAINBOW_POST_WINDOW_US = int(globals().get("RAINBOW_POST_WINDOW_US", default_dark_time_us))


def rainbow_color_segment_for_slot(slot: int, cycle_length: int = 3, labels=None) -> int:
    phase_count = max(1, int(cycle_length))
    slot_index = int(slot) % phase_count
    if labels is None:
        labels = globals().get("display_sequence")
    if labels is not None and slot_index < len(labels):
        try:
            label = int(labels[slot_index])
        except (TypeError, ValueError):
            return slot_index
        if 1 <= label <= phase_count:
            return label - 1
    return slot_index


def analysis_label_for_frame(
    frame_index: int,
    labels=None,
    cycle_length: int = 3,
    label_phase_shift: int = 0,
) -> int:
    phase_count = max(1, int(cycle_length))
    if labels is None:
        labels = globals().get("display_sequence")
    slot = (int(frame_index) - int(label_phase_shift)) % phase_count
    if labels is not None and len(labels) > slot:
        try:
            return int(labels[slot])
        except (TypeError, ValueError):
            pass
    return slot + 1


def frame_title_label_for_frame(
    frame_index: int,
    labels=None,
    cycle_length: int = 3,
    label_phase_shift: int = 0,
) -> int:
    return analysis_label_for_frame(
        frame_index,
        labels=labels,
        cycle_length=cycle_length,
        label_phase_shift=label_phase_shift,
    )


def legend_labels_for_analysis_phase(
    labels=None,
    cycle_length: int = 3,
    label_phase_shift: int = 0,
) -> list[dict[str, int]]:
    phase_count = max(1, int(cycle_length))
    if labels is None:
        labels = globals().get("display_sequence")
    return [
        {
            "slot": int(slot),
            "label": int(analysis_label_for_frame(
                slot,
                labels=labels,
                cycle_length=phase_count,
                label_phase_shift=int(label_phase_shift),
            )),
            "segment": int(rainbow_color_segment_for_slot(slot, phase_count, labels=labels)),
        }
        for slot in range(phase_count)
    ]


def rainbow_cycle_phase(frame_index: int, cycle_length: int = 3, fraction: float = 0.0, labels=None) -> float:
    phase_count = max(1, int(cycle_length))
    slot = int(frame_index) % phase_count
    segment = rainbow_color_segment_for_slot(slot, phase_count, labels=labels)
    frac = max(0.0, min(0.999999, float(fraction)))
    return (segment + frac) / phase_count


def rainbow_phase_color(phase: float, brightness: float = 1.0) -> tuple[int, int, int]:
    position = (float(phase) % 1.0) * 3.0
    segment = int(position) % 3
    frac = position - int(position)
    value = max(0.0, min(1.0, float(brightness)))
    if segment == 0:
        rgb = (1.0 - frac, frac, 0.0)
    elif segment == 1:
        rgb = (0.0, 1.0 - frac, frac)
    else:
        rgb = (frac, 0.0, 1.0 - frac)
    return tuple(int(round(channel * value * 255.0)) for channel in rgb)


def event_cycle_phases(event_ts: np.ndarray, trigger_ts: np.ndarray, cycle_length: int) -> np.ndarray:
    event_ts = np.asarray(event_ts, dtype=np.float64)
    trigger_ts = np.asarray(trigger_ts, dtype=np.float64)
    if event_ts.size == 0 or trigger_ts.size == 0:
        return np.zeros(event_ts.shape, dtype=np.float64)
    trigger_index = np.searchsorted(trigger_ts, event_ts, side="right") - 1
    trigger_index = np.clip(trigger_index, 0, trigger_ts.size - 1)
    next_index = np.clip(trigger_index + 1, 0, trigger_ts.size - 1)
    start_ts = trigger_ts[trigger_index]
    if trigger_ts.size > 1:
        fallback_period = max(float(np.median(np.diff(trigger_ts))), 1.0)
    else:
        fallback_period = max(float(default_window_us), 1.0)
    stop_ts = np.where(next_index > trigger_index, trigger_ts[next_index], start_ts + fallback_period)
    fraction = np.clip((event_ts - start_ts) / np.maximum(stop_ts - start_ts, 1.0), 0.0, 0.999999)
    phase_count = max(1, int(cycle_length))
    trigger_slots = trigger_index % phase_count
    labels = globals().get("display_sequence")
    color_segments = np.asarray(
        [rainbow_color_segment_for_slot(slot, phase_count, labels=labels) for slot in trigger_slots],
        dtype=np.float64,
    )
    return (color_segments + fraction) / phase_count


@lru_cache(maxsize=1)
def sorted_event_arrays_for_provenance():
    timestamps = np.asarray(event_arrays["t"], dtype=np.int64)
    if timestamps.size > 1 and np.any(np.diff(timestamps) < 0):
        order = np.argsort(timestamps, kind="stable")
        return (
            timestamps[order],
            np.asarray(event_arrays["x"], dtype=np.int64)[order],
            np.asarray(event_arrays["y"], dtype=np.int64)[order],
            np.asarray(event_arrays["p"], dtype=np.bool_)[order],
        )
    return (
        timestamps,
        np.asarray(event_arrays["x"], dtype=np.int64),
        np.asarray(event_arrays["y"], dtype=np.int64),
        np.asarray(event_arrays["p"], dtype=np.bool_),
    )


def rainbow_dense_roi(frames: np.ndarray, pad: int, flip_x: bool, crop_mode: str) -> tuple[slice, slice]:
    if str(crop_mode) != "auto":
        return auto_roi(frames, pad, flip_x=flip_x, crop_mode=crop_mode)
    frames_for_roi = frames[:, :, ::-1] if flip_x else frames
    support = np.count_nonzero(np.abs(frames_for_roi) > 0, axis=0).astype(np.float64)
    if not np.any(support):
        return auto_roi(frames, pad, flip_x=flip_x, crop_mode="square")

    def weighted_bounds(profile: np.ndarray, low: float = 0.02, high: float = 0.98) -> tuple[int, int]:
        total = float(np.sum(profile))
        if total <= 0:
            return 0, int(profile.size)
        cumulative = np.cumsum(profile)
        start = int(np.searchsorted(cumulative, total * float(low), side="left"))
        stop = int(np.searchsorted(cumulative, total * float(high), side="right")) + 1
        return max(0, start), min(int(profile.size), max(stop, start + 1))

    y0, y1 = weighted_bounds(np.sum(support, axis=1))
    x0, x1 = weighted_bounds(np.sum(support, axis=0))
    pad = int(pad)
    y0 = max(0, y0 - pad)
    y1 = min(frames.shape[1], y1 + pad)
    x0 = max(0, x0 - pad)
    x1 = min(frames.shape[2], x1 + pad)
    min_span = 48
    if y1 - y0 < min_span:
        center = (y0 + y1) // 2
        y0 = max(0, center - min_span // 2)
        y1 = min(frames.shape[1], y0 + min_span)
    if x1 - x0 < min_span:
        center = (x0 + x1) // 2
        x0 = max(0, center - min_span // 2)
        x1 = min(frames.shape[2], x0 + min_span)
    return slice(y0, y1), slice(x0, x1)



def shape_label_for_frame(frame_index: int, labels=None, cycle_length: int = 3) -> int:
    phase_count = max(1, int(cycle_length))
    slot = int(frame_index) % phase_count
    if labels is None:
        labels = globals().get("display_sequence")
    if labels is not None and len(labels) > slot:
        try:
            return int(labels[slot])
        except (TypeError, ValueError):
            pass
    return slot + 1


def shape_label_color(label: int) -> tuple[int, int, int]:
    colors = {
        1: (255, 70, 70),
        2: (70, 255, 90),
        3: (85, 135, 255),
    }
    return colors.get(int(label), (225, 225, 225))


def shape_template_label_map(
    frames: np.ndarray,
    labels=None,
    cycle_length: int = 3,
    mask_percentile: float = 65.0,
    label_phase_shift: int = 0,
) -> tuple[np.ndarray, dict[int, np.ndarray]]:
    frames = np.asarray(frames, dtype=np.float32)
    if frames.ndim != 3 or frames.shape[0] == 0:
        shape = frames.shape[-2:] if frames.ndim >= 2 else (1, 1)
        return np.zeros(shape, dtype=np.int16), {}
    phase_count = max(1, int(cycle_length))
    if labels is None:
        labels = globals().get("display_sequence")
    if labels is None:
        labels = list(range(1, phase_count + 1))
    label_values = [analysis_label_for_frame(index, labels=labels, cycle_length=phase_count, label_phase_shift=int(label_phase_shift)) for index in range(phase_count)]
    unique_labels = sorted({int(label) for label in label_values})

    templates: dict[int, np.ndarray] = {}
    magnitude = np.abs(frames).astype(np.float32, copy=False)
    for label in unique_labels:
        indices = [
            index
            for index in range(frames.shape[0])
            if analysis_label_for_frame(index, labels=labels, cycle_length=phase_count, label_phase_shift=int(label_phase_shift)) == label
        ]
        if indices:
            templates[label] = np.mean(magnitude[indices], axis=0)
        else:
            templates[label] = np.zeros(frames.shape[1:], dtype=np.float32)

    stack = np.stack([templates[label] for label in unique_labels], axis=0)
    support = np.max(stack, axis=0)
    nonzero = support[support > 0]
    if nonzero.size == 0:
        return np.zeros(frames.shape[1:], dtype=np.int16), templates
    threshold = float(np.percentile(nonzero, max(0.0, min(100.0, float(mask_percentile)))))
    threshold = min(threshold, float(np.max(nonzero)))
    winners = np.argmax(stack, axis=0)
    label_map = np.zeros(frames.shape[1:], dtype=np.int16)
    active = support >= threshold
    for winner_index, label in enumerate(unique_labels):
        label_map[(winners == winner_index) & active] = int(label)
    return label_map, templates


def shape_overlap_scores(signal: np.ndarray, label_map: np.ndarray, labels=None) -> dict[int, float]:
    signal = np.abs(np.asarray(signal, dtype=np.float32))
    label_map = np.asarray(label_map)
    if labels is None:
        labels = sorted(int(label) for label in np.unique(label_map) if int(label) > 0)
    labels = [int(label) for label in labels]
    total = float(np.sum(signal))
    scores = {label: 0.0 for label in labels}
    scores[0] = 0.0
    if total <= 0 or signal.shape != label_map.shape:
        return scores
    assigned = np.zeros(label_map.shape, dtype=bool)
    for label in labels:
        mask = label_map == label
        assigned |= mask
        scores[label] = float(np.sum(signal[mask]) / total)
    scores[0] = float(np.sum(signal[~assigned]) / total)
    return scores


def shape_leak_summary(scores: dict[int, float], expected_label: int) -> tuple[float, int, float]:
    expected_label = int(expected_label)
    own_score = float(scores.get(expected_label, 0.0))
    candidates = [(int(label), float(score)) for label, score in scores.items() if int(label) not in (0, expected_label)]
    if not candidates:
        return own_score, 0, 0.0
    leak_label, leak_score = max(candidates, key=lambda item: item[1])
    return own_score, int(leak_label), float(leak_score)


def shape_leakage_rgb_for_frame(
    frame: np.ndarray,
    label_map: np.ndarray,
    dot_size: int = 1,
    contrast_percentile: float = 99.0,
) -> tuple[np.ndarray, int]:
    signal = np.abs(np.asarray(frame, dtype=np.float32))
    label_map = np.asarray(label_map)
    rgb = np.zeros((*signal.shape, 3), dtype=np.float32)
    active = signal > 0
    if signal.shape != label_map.shape or not np.any(active):
        return rgb.astype(np.uint8), 0
    nonzero = signal[active]
    limit = max(float(np.percentile(nonzero, float(contrast_percentile))), 1.0)
    alpha = np.zeros_like(signal, dtype=np.float32)
    alpha[active] = np.clip(np.log1p(nonzero) / np.log1p(limit), 0.0, 1.0)
    for label in [int(value) for value in np.unique(label_map) if int(value) > 0]:
        mask = label_map == label
        color = np.asarray(shape_label_color(label), dtype=np.float32)
        rgb[mask] = color * alpha[mask, None]
    unassigned = active & (label_map <= 0)
    rgb[unassigned] = np.asarray((115, 115, 115), dtype=np.float32) * alpha[unassigned, None]
    if int(dot_size) > 1:
        for channel in range(3):
            rgb[..., channel] = dilate_plane(rgb[..., channel], int(dot_size))
    return np.clip(rgb, 0, 255).astype(np.uint8), int(np.count_nonzero(active))


def draw_shape_label_legend(draw: ImageDraw.ImageDraw, x: int, y: int, labels) -> None:
    cursor = int(x)
    for label in sorted({int(label) for label in labels}):
        color = shape_label_color(label)
        draw.rectangle((cursor, y, cursor + 22, y + 22), fill=color)
        draw.text((cursor + 28, y - 2), str(label), fill=(220, 220, 220), font=RAINBOW_FONT)
        cursor += 72


def event_provenance_counts_for_frame(
    frame_index: int,
    event_ts: np.ndarray,
    trigger_ts: np.ndarray,
    offset_us: int,
    window_us: int,
    post_window_us: int | None = None,
) -> dict[str, int]:
    event_ts = np.asarray(event_ts, dtype=np.int64)
    trigger_ts = np.asarray(trigger_ts, dtype=np.int64)
    frame_index = int(frame_index)
    if event_ts.size == 0 or trigger_ts.size == 0 or frame_index < 0 or frame_index >= trigger_ts.size:
        return {
            "window_start_us": 0,
            "window_stop_us": 0,
            "selected_events": 0,
            "pre_trigger_events": 0,
            "in_window_events": 0,
            "post_window_events": 0,
            "previous_phase_events": 0,
            "current_phase_events": 0,
            "next_phase_events": 0,
            "other_phase_events": 0,
        }

    trigger_us = int(trigger_ts[frame_index])
    start_us = trigger_us + int(offset_us)
    stop_us = start_us + int(window_us)
    if stop_us < start_us:
        start_us, stop_us = stop_us, start_us
    next_trigger_us = int(trigger_ts[frame_index + 1]) if frame_index + 1 < trigger_ts.size else None
    if post_window_us is None:
        diagnostic_stop_us = next_trigger_us if next_trigger_us is not None else stop_us
    else:
        diagnostic_stop_us = stop_us + max(0, int(post_window_us))
    diagnostic_stop_us = max(stop_us, int(diagnostic_stop_us))

    in_diagnostic = (event_ts >= start_us) & (event_ts < diagnostic_stop_us)
    ts = event_ts[in_diagnostic]
    selected = (ts >= start_us) & (ts < stop_us)
    pre_trigger = selected & (ts < trigger_us)
    in_window = selected & (ts >= trigger_us)
    if next_trigger_us is None:
        post_same_phase_stop = diagnostic_stop_us
    else:
        post_same_phase_stop = min(diagnostic_stop_us, next_trigger_us)
    post_window = (ts >= stop_us) & (ts < post_same_phase_stop)

    source_index = np.searchsorted(trigger_ts, ts, side="right") - 1
    previous_phase = source_index == frame_index - 1
    current_phase = source_index == frame_index
    next_phase = source_index == frame_index + 1
    known_phase = previous_phase | current_phase | next_phase

    return {
        "window_start_us": int(start_us),
        "window_stop_us": int(stop_us),
        "selected_events": int(np.count_nonzero(selected)),
        "pre_trigger_events": int(np.count_nonzero(pre_trigger)),
        "in_window_events": int(np.count_nonzero(in_window)),
        "post_window_events": int(np.count_nonzero(post_window)),
        "previous_phase_events": int(np.count_nonzero(previous_phase)),
        "current_phase_events": int(np.count_nonzero(current_phase)),
        "next_phase_events": int(np.count_nonzero(next_phase)),
        "other_phase_events": int(np.count_nonzero(~known_phase)),
    }


def rainbow_provenance_rgb_for_frame(
    frame_index: int,
    trigger_ts: np.ndarray,
    offset_us: int,
    window_us: int,
    polarity_mode: str,
    y_slice: slice,
    x_slice: slice,
    flip_x: bool,
    dot_size: int,
    contrast_percentile: float,
) -> tuple[np.ndarray, int, int]:
    event_t, event_x, event_y, event_p = sorted_event_arrays_for_provenance()
    H = max(1, int(y_slice.stop - y_slice.start))
    W = max(1, int(x_slice.stop - x_slice.start))
    rgb_sum = np.zeros((H, W, 3), dtype=np.float32)
    weight_sum = np.zeros((H, W), dtype=np.float32)
    if len(trigger_ts) == 0 or int(frame_index) >= len(trigger_ts):
        return np.zeros((H, W, 3), dtype=np.uint8), 0, 0

    start_us = int(trigger_ts[int(frame_index)]) + int(offset_us)
    stop_us = start_us + int(window_us)
    lo = int(np.searchsorted(event_t, start_us, side="left"))
    hi = int(np.searchsorted(event_t, stop_us, side="left"))
    if hi <= lo:
        return np.zeros((H, W, 3), dtype=np.uint8), 0, 0

    ts = event_t[lo:hi]
    xs = event_x[lo:hi]
    ys = event_y[lo:hi]
    ps = event_p[lo:hi]
    mode = str(polarity_mode)
    if mode == "positive":
        keep = ps
        weights = np.ones(int(np.count_nonzero(keep)), dtype=np.float32)
    else:
        keep = np.ones(ts.shape, dtype=bool)
        weights = np.where(ps, 1.0, 0.7).astype(np.float32) if mode == "signed" else np.ones(ts.shape, dtype=np.float32)

    if not np.any(keep):
        return np.zeros((H, W, 3), dtype=np.uint8), int(ts.size), 0
    ts = ts[keep]
    xs = xs[keep]
    ys = ys[keep]
    weights = weights if weights.shape == ts.shape else weights[:ts.shape[0]]
    if flip_x:
        xs = int(width) - 1 - xs
    inside = (ys >= y_slice.start) & (ys < y_slice.stop) & (xs >= x_slice.start) & (xs < x_slice.stop)
    if not np.any(inside):
        return np.zeros((H, W, 3), dtype=np.uint8), int(ts.size), 0

    ts = ts[inside]
    ys = (ys[inside] - y_slice.start).astype(np.int64, copy=False)
    xs = (xs[inside] - x_slice.start).astype(np.int64, copy=False)
    weights = weights[inside]
    phases = event_cycle_phases(ts, trigger_ts, cycle_length)
    colors = np.asarray([rainbow_phase_color(phase) for phase in phases], dtype=np.float32)
    for channel in range(3):
        np.add.at(rgb_sum[..., channel], (ys, xs), colors[:, channel] * weights)
    np.add.at(weight_sum, (ys, xs), weights)

    active = weight_sum > 0
    rgb = np.zeros_like(rgb_sum)
    rgb[active] = rgb_sum[active] / weight_sum[active, None]
    nonzero_weights = weight_sum[active]
    if nonzero_weights.size:
        limit = max(float(np.percentile(nonzero_weights, float(contrast_percentile))), 1.0)
        alpha = np.zeros_like(weight_sum)
        alpha[active] = np.clip(np.log1p(nonzero_weights) / np.log1p(limit), 0.0, 1.0)
        rgb *= alpha[..., None]
    if int(dot_size) > 1:
        for channel in range(3):
            rgb[..., channel] = dilate_plane(rgb[..., channel], int(dot_size))
    return np.clip(rgb, 0, 255).astype(np.uint8), int(ts.size), int(np.count_nonzero(active))


def draw_rainbow_cycle_legend(
    draw: ImageDraw.ImageDraw,
    x: int,
    y: int,
    width_px: int,
    height_px: int,
    label_phase_shift: int = 0,
) -> None:
    width_px = max(1, int(width_px))
    height_px = max(1, int(height_px))
    phase_count = max(1, int(cycle_length))
    for dx in range(width_px):
        color = rainbow_phase_color(dx / max(1, width_px - 1))
        draw.line((x + dx, y, x + dx, y + height_px), fill=color)
    for slot in range(phase_count):
        label = display_sequence[slot] if slot < len(display_sequence) else slot + 1
        lx = x + int((slot + 0.5) * width_px / phase_count) - 4
        draw.text((lx, y + height_px + 3), str(label), fill=(220, 220, 220), font=RAINBOW_FONT)
    if int(label_phase_shift) % phase_count:
        for item in legend_labels_for_analysis_phase(
            labels=display_sequence,
            cycle_length=phase_count,
            label_phase_shift=int(label_phase_shift),
        ):
            lx = x + int((item["slot"] + 0.5) * width_px / phase_count) - 4
            draw.text((lx, y + height_px + 29), str(item["label"]), fill=(255, 220, 120), font=RAINBOW_FONT)


def render_rainbow_provenance_sheet(
    offset_us: int = default_offset_us,
    frame_count: int = 10,
    window_us: int = default_window_us,
    polarity_mode: str = "ignore",
    crop_mode: str = "auto",
    crop_pad: int = 12,
    dot_size: int = 1,
    tile_scale: int = 3,
    focus_frame: int = 1,
    focus_scale: int = 6,
    contrast_percentile: float = 99.0,
    flip_x: bool = True,
    label_phase_shift: int = DEFAULT_RAINBOW_LABEL_PHASE_SHIFT,
    provenance_post_window_us: int = DEFAULT_RAINBOW_POST_WINDOW_US,
):
    start = time.perf_counter()
    requested_frames = max(1, int(frame_count))
    cycles_needed = max(1, ceil((requested_frames + 1) / max(1, int(cycle_length))))
    frames, trigger_ts, alignment, cycle_info = accumulate_cached(
        int(offset_us), int(cycles_needed), int(window_us), str(polarity_mode)
    )
    visible_count = min(requested_frames, int(frames.shape[0]))
    frames = frames[:visible_count]
    trigger_ts = np.asarray(trigger_ts, dtype=np.int64)
    visible_trigger_ts = trigger_ts[:visible_count]
    y_slice, x_slice = rainbow_dense_roi(frames, crop_pad, flip_x=flip_x, crop_mode=crop_mode)
    cols = max(1, int(cycle_length))
    rows = max(1, ceil(max(1, visible_count) / cols))

    tiles = []
    event_counts = []
    active_pixels = []
    provenance_counts = []
    event_t_for_counts, _, _, _ = sorted_event_arrays_for_provenance()
    for index in range(visible_count):
        rgb, events_used, active = rainbow_provenance_rgb_for_frame(
            index,
            trigger_ts,
            int(offset_us),
            int(window_us),
            str(polarity_mode),
            y_slice,
            x_slice,
            bool(flip_x),
            int(dot_size),
            float(contrast_percentile),
        )
        tiles.append(resize_nearest(rgb, int(tile_scale)))
        event_counts.append(events_used)
        active_pixels.append(active)
        provenance_counts.append(event_provenance_counts_for_frame(
            index,
            event_t_for_counts,
            trigger_ts,
            int(offset_us),
            int(window_us),
            post_window_us=int(provenance_post_window_us),
        ))

    if not tiles:
        tiles = [Image.new("RGB", (16, 16), (0, 0, 0))]
    tile_w = tiles[0].width
    tile_h = tiles[0].height
    label_h = 92
    header_h = 220
    gap = 8
    focus_index = max(0, min(visible_count - 1, int(focus_frame) - 1)) if visible_count else 0
    focus_rgb, _, _ = rainbow_provenance_rgb_for_frame(
        focus_index,
        trigger_ts,
        int(offset_us),
        int(window_us),
        str(polarity_mode),
        y_slice,
        x_slice,
        bool(flip_x),
        int(dot_size),
        float(contrast_percentile),
    )
    focus_image = resize_nearest(focus_rgb, int(focus_scale))

    sheet_w = cols * tile_w + (cols - 1) * gap
    sheet_h = rows * (tile_h + label_h) + (rows - 1) * gap
    canvas_w = max(sheet_w, focus_image.width, 520) + 20
    canvas_h = header_h + focus_image.height + 22 + sheet_h + 24
    canvas = Image.new("RGB", (canvas_w, canvas_h), (14, 14, 14))
    draw = ImageDraw.Draw(canvas)
    first_trigger = int(visible_trigger_ts[0]) if visible_trigger_ts.size else 0
    title = (
        f"rainbow provenance | offset {int(offset_us)} us | window {int(window_us)} us | "
        f"first {visible_count} frames | {polarity_mode} | dot={int(dot_size)}"
    )
    draw.text((10, 8), title, fill=(235, 235, 235), font=RAINBOW_FONT)
    draw.text(
        (10, 42),
        "hue = event timestamp in repeating trigger cycle; brightness = event count at pixel",
        fill=(180, 180, 180),
        font=RAINBOW_FONT,
    )
    draw.text((10, 76), f"roi y={y_slice.start}:{y_slice.stop} x={x_slice.start}:{x_slice.stop}", fill=(180, 180, 180), font=RAINBOW_FONT)
    draw.text((10, 108), f"raw labels={display_sequence}; analysis phase shift={int(label_phase_shift) % max(1, int(cycle_length))}", fill=(255, 220, 120), font=RAINBOW_FONT)
    draw_rainbow_cycle_legend(draw, 10, 140, min(420, canvas_w - 30), 14, label_phase_shift=int(label_phase_shift))

    focus_y = header_h
    canvas.paste(focus_image, (10, focus_y))
    draw.rectangle((10, focus_y, 10 + focus_image.width - 1, focus_y + focus_image.height - 1), outline=(255, 245, 170))
    focus_raw_label = display_sequence[focus_index % cols] if display_sequence else focus_index % cols + 1
    focus_label = frame_title_label_for_frame(focus_index, labels=display_sequence, cycle_length=cols, label_phase_shift=int(label_phase_shift))
    draw.text((16, focus_y + 6), f"focus {focus_index + 1}: analysis {focus_label} raw {focus_raw_label}", fill=(255, 245, 170), font=RAINBOW_FONT)

    grid_y = focus_y + focus_image.height + 26
    for index, image in enumerate(tiles[:visible_count]):
        row = index // cols
        col = index % cols
        x = 10 + col * (tile_w + gap)
        y = grid_y + row * (tile_h + label_h + gap)
        canvas.paste(image, (x, y + label_h))
        slot = index % cols
        cycle = index // cols + 1
        raw_label = display_sequence[slot] if slot < len(display_sequence) else slot + 1
        label = frame_title_label_for_frame(index, labels=display_sequence, cycle_length=cols, label_phase_shift=int(label_phase_shift))
        dt = int(visible_trigger_ts[index] - first_trigger) if index < len(visible_trigger_ts) else 0
        color = (255, 245, 170) if index == focus_index else shape_label_color(label)
        counts = provenance_counts[index]
        draw.text((x, y), f"{index + 1}: {label} raw {raw_label} c{cycle} +{dt}us", fill=color, font=RAINBOW_FONT)
        draw.text((x, y + 30), f"ev={event_counts[index]} px={active_pixels[index]} pre={counts['pre_trigger_events']} in={counts['in_window_events']}", fill=(170, 170, 170), font=RAINBOW_FONT)
        draw.text((x, y + 60), f"prev={counts['previous_phase_events']} cur={counts['current_phase_events']} next={counts['next_phase_events']} post={counts['post_window_events']}", fill=(140, 140, 140), font=RAINBOW_FONT)

    elapsed_ms = (time.perf_counter() - start) * 1000
    draw.text((canvas_w - 115, 8), f"{elapsed_ms:.0f} ms", fill=(120, 210, 120), font=RAINBOW_FONT)
    return canvas, frames, event_counts, alignment, cycle_info



def render_shape_leakage_sheet(
    offset_us: int = default_offset_us,
    frame_count: int = 10,
    window_us: int = default_window_us,
    polarity_mode: str = "ignore",
    crop_mode: str = "auto",
    crop_pad: int = 12,
    dot_size: int = 1,
    tile_scale: int = 3,
    focus_frame: int = 1,
    focus_scale: int = 6,
    contrast_percentile: float = 99.0,
    flip_x: bool = True,
    label_phase_shift: int = DEFAULT_RAINBOW_LABEL_PHASE_SHIFT,
    mask_percentile: float = 65.0,
):
    start = time.perf_counter()
    requested_frames = max(1, int(frame_count))
    phase_count = max(1, int(cycle_length))
    cycles_needed = max(1, ceil((requested_frames + 1) / phase_count))
    frames, trigger_ts, alignment, cycle_info = accumulate_cached(
        int(offset_us), int(cycles_needed), int(window_us), str(polarity_mode)
    )
    visible_count = min(requested_frames, int(frames.shape[0]))
    frames = frames[:visible_count]
    trigger_ts = np.asarray(trigger_ts, dtype=np.int64)
    visible_trigger_ts = trigger_ts[:visible_count]
    y_slice, x_slice = rainbow_dense_roi(frames, crop_pad, flip_x=flip_x, crop_mode=crop_mode)
    cropped = frames[:, y_slice, x_slice]
    if bool(flip_x):
        cropped = cropped[:, :, ::-1]
    labels = [frame_title_label_for_frame(index, labels=display_sequence, cycle_length=phase_count, label_phase_shift=int(label_phase_shift)) for index in range(phase_count)]
    score_labels = sorted({int(label) for label in labels})
    label_map, templates = shape_template_label_map(
        cropped,
        labels=display_sequence,
        cycle_length=phase_count,
        mask_percentile=float(mask_percentile),
        label_phase_shift=max(0, int(label_phase_shift) - 1),
    )
    cols = phase_count
    rows = max(1, ceil(max(1, visible_count) / cols))

    tiles = []
    scores_by_frame = []
    activity_counts = []
    active_pixels = []
    for index in range(visible_count):
        signal = np.abs(cropped[index])
        scores = shape_overlap_scores(signal, label_map, labels=score_labels)
        rgb, active = shape_leakage_rgb_for_frame(
            cropped[index],
            label_map,
            dot_size=int(dot_size),
            contrast_percentile=float(contrast_percentile),
        )
        tiles.append(resize_nearest(rgb, int(tile_scale)))
        scores_by_frame.append(scores)
        activity_counts.append(int(round(float(np.sum(signal)))))
        active_pixels.append(active)

    if not tiles:
        tiles = [Image.new("RGB", (16, 16), (0, 0, 0))]
    tile_w = tiles[0].width
    tile_h = tiles[0].height
    label_h = 88
    header_h = 150
    gap = 8
    focus_index = max(0, min(visible_count - 1, int(focus_frame) - 1)) if visible_count else 0
    focus_rgb, _ = shape_leakage_rgb_for_frame(
        cropped[focus_index] if visible_count else np.zeros((1, 1), dtype=np.float32),
        label_map,
        dot_size=int(dot_size),
        contrast_percentile=float(contrast_percentile),
    )
    focus_image = resize_nearest(focus_rgb, int(focus_scale))

    sheet_w = cols * tile_w + (cols - 1) * gap
    sheet_h = rows * (tile_h + label_h) + (rows - 1) * gap
    canvas_w = max(sheet_w, focus_image.width, 650) + 20
    canvas_h = header_h + focus_image.height + 22 + sheet_h + 24
    canvas = Image.new("RGB", (canvas_w, canvas_h), (14, 14, 14))
    draw = ImageDraw.Draw(canvas)
    first_trigger = int(visible_trigger_ts[0]) if visible_trigger_ts.size else 0
    title = (
        f"shape overlap leakage | offset {int(offset_us)} us | window {int(window_us)} us | "
        f"first {visible_count} frames | mask p{float(mask_percentile):.0f} | {polarity_mode}"
    )
    draw.text((10, 8), title, fill=(235, 235, 235), font=RAINBOW_FONT)
    draw.text(
        (10, 42),
        "pixel color = strongest digit template owner; own/leak = activity share in expected/off-label masks",
        fill=(180, 180, 180),
        font=RAINBOW_FONT,
    )
    draw.text((10, 76), f"roi y={y_slice.start}:{y_slice.stop} x={x_slice.start}:{x_slice.stop}", fill=(180, 180, 180), font=RAINBOW_FONT)
    draw_shape_label_legend(draw, 10, 112, score_labels)

    focus_y = header_h
    canvas.paste(focus_image, (10, focus_y))
    draw.rectangle((10, focus_y, 10 + focus_image.width - 1, focus_y + focus_image.height - 1), outline=(255, 245, 170))
    focus_label = frame_title_label_for_frame(focus_index, labels=display_sequence, cycle_length=phase_count, label_phase_shift=int(label_phase_shift))
    focus_scores = scores_by_frame[focus_index] if scores_by_frame else {label: 0.0 for label in score_labels}
    own_score, leak_label, leak_score = shape_leak_summary(focus_scores, focus_label)
    draw.text(
        (16, focus_y + 6),
        f"focus {focus_index + 1}: exp {focus_label} own={own_score * 100:.0f}% leak {leak_label}:{leak_score * 100:.0f}%",
        fill=(255, 245, 170),
        font=RAINBOW_FONT,
    )

    grid_y = focus_y + focus_image.height + 26
    for index, image in enumerate(tiles[:visible_count]):
        row = index // cols
        col = index % cols
        x = 10 + col * (tile_w + gap)
        y = grid_y + row * (tile_h + label_h + gap)
        canvas.paste(image, (x, y + label_h))
        cycle = index // cols + 1
        label = frame_title_label_for_frame(index, labels=display_sequence, cycle_length=phase_count, label_phase_shift=int(label_phase_shift))
        dt = int(visible_trigger_ts[index] - first_trigger) if index < len(visible_trigger_ts) else 0
        own_score, leak_label, leak_score = shape_leak_summary(scores_by_frame[index], label)
        color = (255, 245, 170) if index == focus_index else shape_label_color(label)
        draw.text((x, y), f"{index + 1}: exp {label} c{cycle} +{dt}us", fill=color, font=RAINBOW_FONT)
        draw.text((x, y + 30), f"own={own_score * 100:.0f}% leak {leak_label}:{leak_score * 100:.0f}%", fill=(190, 190, 190), font=RAINBOW_FONT)
        draw.text((x, y + 60), f"act={activity_counts[index]} px={active_pixels[index]}", fill=(140, 140, 140), font=RAINBOW_FONT)

    elapsed_ms = (time.perf_counter() - start) * 1000
    draw.text((canvas_w - 115, 8), f"{elapsed_ms:.0f} ms", fill=(120, 210, 120), font=RAINBOW_FONT)
    return canvas, frames, scores_by_frame, alignment, cycle_info



def render_static_rainbow_diagnostic(
    view: str | None = None,
    frame_count: int | None = None,
    offset_us: int = default_offset_us,
    window_us: int = default_window_us,
    polarity_mode: str = "ignore",
    crop_mode: str = "auto",
    crop_pad: int = 12,
    dot_size: int = 1,
    tile_scale: int = 3,
    focus_frame: int = 1,
    focus_scale: int = 6,
    contrast_percentile: float = 99.0,
    flip_x: bool = True,
    label_phase_shift: int = DEFAULT_RAINBOW_LABEL_PHASE_SHIFT,
    mask_percentile: float = 65.0,
):
    selected_view = str(STATIC_RAINBOW_VIEW if view is None else view).strip().lower()
    selected_frames = STATIC_RAINBOW_FRAME_COUNT if frame_count is None else int(frame_count)
    common_kwargs = dict(
        offset_us=int(offset_us),
        frame_count=max(1, int(selected_frames)),
        window_us=int(window_us),
        polarity_mode=str(polarity_mode),
        crop_mode=str(crop_mode),
        crop_pad=int(crop_pad),
        dot_size=int(dot_size),
        tile_scale=int(tile_scale),
        focus_frame=int(focus_frame),
        focus_scale=int(focus_scale),
        contrast_percentile=float(contrast_percentile),
        flip_x=bool(flip_x),
        label_phase_shift=int(label_phase_shift),
    )
    if selected_view in {"shape", "shape overlap", "overlap", "leakage"}:
        return render_shape_leakage_sheet(**common_kwargs, mask_percentile=float(mask_percentile))
    return render_rainbow_provenance_sheet(**common_kwargs)


if HAVE_WIDGETS and ENABLE_INTERACTIVE_WIDGETS:
    rainbow_frame_slider = widgets.IntSlider(
        value=max(1, min(STATIC_RAINBOW_FRAME_COUNT, int(available_cycles) * int(cycle_length))),
        min=1,
        max=max(1, min(60, int(available_cycles) * int(cycle_length))),
        step=1,
        description="frames",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    rainbow_offset_slider = widgets.IntSlider(
        value=default_offset_us,
        min=offset_min_us,
        max=offset_max_us,
        step=25,
        description="offset us",
        continuous_update=True,
        layout=widgets.Layout(width="560px"),
    )
    rainbow_window_slider = widgets.IntSlider(
        value=default_window_us,
        min=100,
        max=max(default_window_us * 5, default_dark_time_us + default_window_us + 1000),
        step=100,
        description="window us",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    rainbow_focus_slider = widgets.IntSlider(value=1, min=1, max=rainbow_frame_slider.value, step=1, description="focus frame", continuous_update=False, layout=widgets.Layout(width="560px"))
    rainbow_phase_shift_slider = widgets.IntSlider(value=DEFAULT_RAINBOW_LABEL_PHASE_SHIFT, min=0, max=max(0, int(cycle_length) - 1), step=1, description="label phase", continuous_update=False, layout=widgets.Layout(width="560px"))
    rainbow_view_dropdown = widgets.Dropdown(value="timestamp", options=[("timestamp colors", "timestamp"), ("shape overlap", "shape")], description="view")
    rainbow_mask_slider = widgets.FloatSlider(value=65.0, min=10.0, max=95.0, step=1.0, description="mask %", continuous_update=False, layout=widgets.Layout(width="560px"))
    rainbow_polarity_dropdown = widgets.Dropdown(value="ignore", options=["ignore", "signed", "positive"], description="accum")
    rainbow_crop_dropdown = widgets.Dropdown(value="auto", options=["auto", "square", "full"], description="crop")
    rainbow_percentile_slider = widgets.FloatSlider(value=99.0, min=90.0, max=100.0, step=0.1, description="event vmax %", continuous_update=False, layout=widgets.Layout(width="560px"))
    rainbow_pad_slider = widgets.IntSlider(value=12, min=0, max=80, step=2, description="crop pad", continuous_update=False, layout=widgets.Layout(width="560px"))
    rainbow_dot_slider = widgets.IntSlider(value=1, min=1, max=8, step=1, description="dot size", continuous_update=True, layout=widgets.Layout(width="560px"))
    rainbow_tile_scale_slider = widgets.IntSlider(value=3, min=1, max=6, step=1, description="tile scale", continuous_update=False, layout=widgets.Layout(width="560px"))
    rainbow_focus_scale_slider = widgets.IntSlider(value=6, min=2, max=10, step=1, description="focus scale", continuous_update=False, layout=widgets.Layout(width="560px"))
    rainbow_flip_checkbox = widgets.Checkbox(value=True, description="flip x for viewing")
    rainbow_out = widgets.Output()

    def sync_rainbow_focus_max(*_):
        rainbow_focus_slider.max = max(1, int(rainbow_frame_slider.value))
        if rainbow_focus_slider.value > rainbow_focus_slider.max:
            rainbow_focus_slider.value = rainbow_focus_slider.max

    def redraw_rainbow(_=None):
        sync_rainbow_focus_max()
        with rainbow_out:
            clear_output(wait=True)
            common_kwargs = dict(
                offset_us=rainbow_offset_slider.value,
                frame_count=rainbow_frame_slider.value,
                window_us=rainbow_window_slider.value,
                polarity_mode=rainbow_polarity_dropdown.value,
                crop_mode=rainbow_crop_dropdown.value,
                crop_pad=rainbow_pad_slider.value,
                dot_size=rainbow_dot_slider.value,
                tile_scale=rainbow_tile_scale_slider.value,
                focus_frame=rainbow_focus_slider.value,
                focus_scale=rainbow_focus_scale_slider.value,
                contrast_percentile=rainbow_percentile_slider.value,
                flip_x=rainbow_flip_checkbox.value,
                label_phase_shift=rainbow_phase_shift_slider.value,
            )
            if rainbow_view_dropdown.value == "shape":
                image, frames, event_counts, alignment, cycle_info = render_shape_leakage_sheet(
                    **common_kwargs,
                    mask_percentile=rainbow_mask_slider.value,
                )
            else:
                image, frames, event_counts, alignment, cycle_info = render_rainbow_provenance_sheet(**common_kwargs)
            display(DisplayImage(data=png_bytes(image)))

    for widget in [
        rainbow_frame_slider,
        rainbow_offset_slider,
        rainbow_window_slider,
        rainbow_focus_slider,
        rainbow_view_dropdown,
        rainbow_phase_shift_slider,
        rainbow_polarity_dropdown,
        rainbow_crop_dropdown,
        rainbow_percentile_slider,
        rainbow_mask_slider,
        rainbow_pad_slider,
        rainbow_dot_slider,
        rainbow_tile_scale_slider,
        rainbow_focus_scale_slider,
        rainbow_flip_checkbox,
    ]:
        widget.observe(redraw_rainbow, names="value")

    rainbow_controls = widgets.VBox([
        rainbow_frame_slider,
        rainbow_offset_slider,
        rainbow_window_slider,
        rainbow_focus_slider,
        rainbow_phase_shift_slider,
        widgets.HBox([rainbow_view_dropdown, rainbow_polarity_dropdown, rainbow_crop_dropdown, rainbow_flip_checkbox]),
        widgets.HBox([rainbow_percentile_slider, rainbow_mask_slider]),
        rainbow_pad_slider,
        widgets.HBox([rainbow_dot_slider, rainbow_tile_scale_slider, rainbow_focus_scale_slider]),
    ])
    display(rainbow_controls)
    display(rainbow_out)
    redraw_rainbow()
else:
    image, frames, event_counts, alignment, cycle_info = render_static_rainbow_diagnostic()
    display(DisplayImage(data=png_bytes(image)))

## Interactive Timeline and Bleed Controls

Use this when the frame grid looks bad and you want to separate timing problems from image-rendering problems. These controls update on release instead of continuously, which keeps the notebook responsive.

In [ ]:
try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ModuleNotFoundError:
    HAVE_WIDGETS = False


ENABLE_INTERACTIVE_WIDGETS = bool(globals().get("ENABLE_INTERACTIVE_WIDGETS", True))

if HAVE_WIDGETS and ENABLE_INTERACTIVE_WIDGETS:
    diag_offset_slider = widgets.IntSlider(
        value=default_offset_us,
        min=offset_min_us,
        max=offset_max_us,
        step=25,
        description="offset us",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    diag_window_slider = widgets.IntSlider(
        value=default_window_us,
        min=100,
        max=max(2500, default_window_us * 2),
        step=25,
        description="window us",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    diag_cycles_slider = widgets.IntSlider(
        value=default_cycles,
        min=1,
        max=max(1, min(20, recording.stats["trigger_edges"].get("rising", 0) // max(1, cycle_length))),
        step=1,
        description="cycles",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    diag_bin_slider = widgets.IntSlider(value=100, min=25, max=500, step=25, description="time bin", continuous_update=False, layout=widgets.Layout(width="560px"))
    diag_pre_slider = widgets.IntSlider(value=1000, min=0, max=3000, step=100, description="pre us", continuous_update=False, layout=widgets.Layout(width="560px"))
    diag_post_default_us = max(3000, default_window_us + 1000)
    diag_post_slider = widgets.IntSlider(value=diag_post_default_us, min=500, max=max(6000, default_window_us * 2), step=100, description="post us", continuous_update=False, layout=widgets.Layout(width="560px"))
    diag_mask_slider = widgets.IntSlider(value=3, min=1, max=9, step=1, description="mask dilate", continuous_update=False, layout=widgets.Layout(width="560px"))
    diag_polarity_dropdown = widgets.Dropdown(value="ignore", options=["ignore", "signed", "positive"], description="accum")
    diag_out = widgets.Output()

    def redraw_diagnostics(_=None):
        with diag_out:
            clear_output(wait=True)
            timeline_image = render_trigger_timeline(
                offset_us=diag_offset_slider.value,
                cycles=diag_cycles_slider.value,
                window_us=diag_window_slider.value,
                bin_us=diag_bin_slider.value,
                polarity_mode=diag_polarity_dropdown.value,
            )
            activity_image = render_relative_activity(
                offset_us=diag_offset_slider.value,
                cycles=diag_cycles_slider.value,
                window_us=diag_window_slider.value,
                pre_us=diag_pre_slider.value,
                post_us=diag_post_slider.value,
                bin_us=max(25, diag_bin_slider.value // 2),
                polarity_mode=diag_polarity_dropdown.value,
            )
            bleed_image = render_bleed_matrix(
                offset_us=diag_offset_slider.value,
                cycles=diag_cycles_slider.value,
                window_us=diag_window_slider.value,
                polarity_mode=diag_polarity_dropdown.value,
                mask_dilate=diag_mask_slider.value,
            )
            display(DisplayImage(data=png_bytes(timeline_image)))
            display(DisplayImage(data=png_bytes(activity_image)))
            display(DisplayImage(data=png_bytes(bleed_image)))

    for widget in [
        diag_offset_slider,
        diag_window_slider,
        diag_cycles_slider,
        diag_bin_slider,
        diag_pre_slider,
        diag_post_slider,
        diag_mask_slider,
        diag_polarity_dropdown,
    ]:
        widget.observe(redraw_diagnostics, names="value")

    display(widgets.VBox([
        diag_offset_slider,
        diag_window_slider,
        diag_cycles_slider,
        widgets.HBox([diag_bin_slider, diag_mask_slider, diag_polarity_dropdown]),
        widgets.HBox([diag_pre_slider, diag_post_slider]),
    ]))
    display(diag_out)
    redraw_diagnostics()
else:
    timeline_image = render_trigger_timeline(offset_us=default_offset_us)
    activity_image = render_relative_activity(offset_us=default_offset_us)
    bleed_image = render_bleed_matrix(offset_us=default_offset_us)
    display(DisplayImage(data=png_bytes(timeline_image)))
    display(DisplayImage(data=png_bytes(activity_image)))
    display(DisplayImage(data=png_bytes(bleed_image)))


## Offset Score Table

This is a quick numerical scan. It does not pick a cycle or reorder frames; it only reports how many nonzero pixels each offset produces in the first `N x 5` ordered frames.

In [ ]:
def offset_score_table(offsets=range(100, 501, 50), cycles=default_cycles, window_us=1750, polarity_mode="ignore"):
    rows = []
    for offset in offsets:
        frames, trigger_ts, alignment, cycle_info = accumulate_cached(int(offset), int(cycles), int(window_us), str(polarity_mode))
        counts = np.count_nonzero(np.abs(frames) > 0, axis=(1, 2))
        first_cycle = counts[:cycle_length] if counts.size else np.array([], dtype=int)
        rows.append({
            "offset_us": int(offset),
            "frame1": int(counts[0]) if counts.size else 0,
            "first_cycle_min": int(first_cycle.min()) if first_cycle.size else 0,
            "first_cycle_median": int(np.median(first_cycle)) if first_cycle.size else 0,
            "all_min": int(counts.min()) if counts.size else 0,
            "all_median": int(np.median(counts)) if counts.size else 0,
            "all_max": int(counts.max()) if counts.size else 0,
        })
    return sorted(rows, key=lambda row: (row["first_cycle_min"], row["frame1"], row["all_median"]), reverse=True)


scores = offset_score_table()
for row in scores[:12]:
    print(row)

## Automatic Offset/Window Scan

This ranks `(accumulation_start_offset_us, window_us)` pairs without reordering triggers. The balanced score is a starting point; compare it with the signal-heavy and low-bleed lists before choosing a value for a real run.

In [ ]:
# def _dilate_mask(mask: np.ndarray, size: int = 3) -> np.ndarray:
#     if int(size) <= 1:
#         return mask.astype(bool, copy=False)
#     return dilate_plane(mask.astype(np.float32), int(size)) > 0
#
#
# def _frame_labels_for_shift(frame_count: int, label_phase_shift: int = 0) -> list[int]:
#     labels = display_sequence or list(range(1, max(1, int(cycle_length)) + 1))
#     phase_count = max(1, int(cycle_length))
#     shift = int(label_phase_shift) % phase_count
#     return [int(labels[(index + shift) % phase_count]) for index in range(int(frame_count))]
#
#
# def _repeatability_separation(frames: np.ndarray, label_phase_shift: int = 0) -> tuple[float, float, float]:
#     mag = np.abs(np.asarray(frames, dtype=np.float32))
#     if mag.ndim != 3 or mag.shape[0] < 2:
#         return 0.0, 0.0, 0.0
#     flat = mag.reshape((mag.shape[0], -1))
#     norms = np.linalg.norm(flat, axis=1) + 1e-9
#     labels = _frame_labels_for_shift(flat.shape[0], label_phase_shift=label_phase_shift)
#     same = []
#     different = []
#     for i in range(flat.shape[0]):
#         for j in range(i + 1, flat.shape[0]):
#             sim = float(flat[i].dot(flat[j]) / (norms[i] * norms[j]))
#             if labels[i] == labels[j]:
#                 same.append(sim)
#             else:
#                 different.append(sim)
#     same_mean = float(np.mean(same)) if same else 0.0
#     different_mean = float(np.mean(different)) if different else 0.0
#     return same_mean, different_mean, same_mean - different_mean
#
#
# def _top_fraction_concentration(frames: np.ndarray, fraction: float = 0.05) -> float:
#     mag = np.abs(np.asarray(frames, dtype=np.float32))
#     values = mag.reshape((mag.shape[0], -1))
#     concentrations = []
#     for row in values:
#         active = row[row > 0]
#         if active.size == 0:
#             concentrations.append(0.0)
#             continue
#         k = max(1, int(active.size * float(fraction)))
#         top = np.partition(active, -k)[-k:]
#         concentrations.append(float(top.sum() / max(float(active.sum()), 1e-9)))
#     return float(np.mean(concentrations)) if concentrations else 0.0
#
#
# def _relative_event_count(trigger_ts: np.ndarray, start_offset_us: int, stop_offset_us: int, polarity_mode: str = "ignore") -> int:
#     event_t, event_x, event_y, event_p = sorted_event_arrays_for_provenance()
#     trigger_ts = np.asarray(trigger_ts, dtype=np.int64)
#     total = 0
#     for trigger in trigger_ts:
#         lo = int(np.searchsorted(event_t, int(trigger) + int(start_offset_us), side="left"))
#         hi = int(np.searchsorted(event_t, int(trigger) + int(stop_offset_us), side="left"))
#         if hi <= lo:
#             continue
#         if str(polarity_mode) == "positive":
#             total += int(np.count_nonzero(event_p[lo:hi]))
#         else:
#             total += int(hi - lo)
#     return total
#
#
# def phase_candidate_metrics(
#     offset_us: int,
#     window_us: int,
#     cycles: int = default_cycles,
#     polarity_mode: str = "ignore",
#     label_phase_shift: int = 0,
# ) -> dict:
#     frames, trigger_ts, alignment, cycle_info = accumulate_cached(
#         int(offset_us), int(cycles), int(window_us), str(polarity_mode)
#     )
#     usable = (frames.shape[0] // max(1, int(cycle_length))) * max(1, int(cycle_length))
#     frames = frames[:usable]
#     trigger_ts = np.asarray(trigger_ts[:usable], dtype=np.int64)
#     if frames.size == 0 or usable == 0:
#         return {"valid": False, "offset_us": int(offset_us), "window_us": int(window_us)}
#
#     mag = np.abs(frames.astype(np.float32, copy=False))
#     signal = mag.reshape((mag.shape[0], -1)).sum(axis=1)
#     active = (mag > 0).reshape((mag.shape[0], -1)).sum(axis=1)
#     same_mean, different_mean, label_separation = _repeatability_separation(
#         frames, label_phase_shift=int(label_phase_shift)
#     )
#     concentration = _top_fraction_concentration(frames, fraction=0.05)
#
#     pre_dark_events = _relative_event_count(
#         trigger_ts, int(offset_us) - int(default_dark_time_us), int(offset_us), polarity_mode=str(polarity_mode)
#     )
#     post_dark_events = _relative_event_count(
#         trigger_ts,
#         int(offset_us) + int(window_us),
#         int(offset_us) + int(window_us) + int(default_dark_time_us),
#         polarity_mode=str(polarity_mode),
#     )
#     window_events = max(1.0, float(np.sum(signal)))
#     dark_tail_penalty = float((pre_dark_events + post_dark_events) / window_events)
#
#     spacing = np.diff(trigger_ts).astype(np.float32)
#     min_gap_after_window = float(np.min(spacing - int(window_us))) if spacing.size else 9999.0
#     overlap_penalty = max(0.0, -min_gap_after_window / 250.0)
#     balanced_score = (
#         4.0 * float(label_separation)
#         + 1.5 * float(concentration)
#         + 0.20 * np.log1p(float(np.median(signal)))
#         - 1.25 * float(dark_tail_penalty)
#         - overlap_penalty
#     )
#
#     labels = _frame_labels_for_shift(frames.shape[0], label_phase_shift=int(label_phase_shift))
#     label_signal = {
#         int(label): float(np.mean([signal[index] for index, value in enumerate(labels) if value == label]))
#         for label in sorted(set(labels))
#     }
#     return {
#         "valid": True,
#         "offset_us": int(offset_us),
#         "window_us": int(window_us),
#         "label_phase_shift": int(label_phase_shift),
#         "balanced_score": float(balanced_score),
#         "label_separation": float(label_separation),
#         "same_label_similarity": float(same_mean),
#         "between_label_similarity": float(different_mean),
#         "top5_concentration": float(concentration),
#         "dark_tail_penalty": float(dark_tail_penalty),
#         "median_signal": float(np.median(signal)),
#         "median_active": int(np.median(active)),
#         "pre_dark_events": int(pre_dark_events),
#         "post_dark_events": int(post_dark_events),
#         "min_gap_after_window_us": int(round(min_gap_after_window)),
#         "label_signal": label_signal,
#     }
#
#
# def run_phase_offset_scan(
#     offsets=range(-int(trigger_period_us + 1000), int(trigger_period_us + 1000) + 1, 250),
#     windows=(3000, 3500, 4000, 4500, 5000),
#     cycles=default_cycles,
#     polarity_mode="ignore",
# ):
#     started = time.perf_counter()
#     rows = []
#     phase_count = max(1, int(cycle_length))
#     for window_us in windows:
#         for offset_us in offsets:
#             for label_phase_shift in range(phase_count):
#                 row = phase_candidate_metrics(
#                     offset_us,
#                     window_us,
#                     cycles=cycles,
#                     polarity_mode=polarity_mode,
#                     label_phase_shift=label_phase_shift,
#                 )
#                 if row.get("valid"):
#                     rows.append(row)
#     print(f"scanned {len(rows)} phase/offset candidates in {time.perf_counter() - started:.1f}s")
#     return rows
#
#
# def print_phase_ranked(rows, key: str = "balanced_score", title: str = "Phase/offset candidates", limit: int = 12):
#     print("\n" + title)
#     print("offset window phase score sep same diff conc dark med labels")
#     for row in sorted(rows, key=lambda item: item[key], reverse=True)[:limit]:
#         label_text = ",".join(f"{label}:{value:.0f}" for label, value in sorted(row["label_signal"].items()))
#         print(
#             f"{row['offset_us']:>6} {row['window_us']:>6} {row['label_phase_shift']:>5} "
#             f"{row[key]:>6.3f} {row['label_separation']:>5.3f} "
#             f"{row['same_label_similarity']:>5.3f} {row['between_label_similarity']:>5.3f} "
#             f"{row['top5_concentration']:>5.3f} {row['dark_tail_penalty']:>5.3f} "
#             f"{row['median_signal']:>5.0f} {label_text}"
#         )
#
#
# phase_scan_rows = run_phase_offset_scan()
# print_phase_ranked(phase_scan_rows, "balanced_score", "Balanced phase/offset candidates")


## Trim Displayed Trigger Window AEDAT4

Write a compact AEDAT4 file containing only the events around the displayed trigger frames. The default output keeps the same 30 rising triggers used by the static rainbow view and includes a 2000 us timestamp buffer before the first trigger and after the last trigger.


In [ ]:
TRIMMED_AEDAT4_TRIGGER_COUNT = STATIC_RAINBOW_FRAME_COUNT
TRIMMED_AEDAT4_BUFFER_US = 2000
TRIMMED_AEDAT4_PATH = RUN_DIR / "trimmed_displayed_30_triggers_plus_2000us.aedat4"
TRIMMED_AEDAT4_METADATA_PATH = RUN_DIR / "trimmed_displayed_30_triggers_plus_2000us.json"


def displayed_trigger_trim_window(trigger_ts, frame_count: int = 30, buffer_us: int = 2000):
    trigger_ts = np.asarray(trigger_ts, dtype=np.int64)
    if trigger_ts.size == 0:
        raise ValueError("at least one trigger timestamp is required")
    selected_count = max(1, min(int(frame_count), int(trigger_ts.size)))
    selected = trigger_ts[:selected_count]
    buffer_us = max(0, int(buffer_us))
    first_trigger_us = int(selected[0])
    last_trigger_us = int(selected[-1])
    return {
        "start_us": int(first_trigger_us - buffer_us),
        "stop_us": int(last_trigger_us + buffer_us),
        "first_trigger_us": first_trigger_us,
        "last_trigger_us": last_trigger_us,
        "selected_trigger_count": int(selected_count),
        "buffer_us": int(buffer_us),
    }


def trim_event_arrays_for_time_window(arrays, window):
    timestamps = np.asarray(arrays["t"], dtype=np.int64)
    keep = (timestamps >= int(window["start_us"])) & (timestamps < int(window["stop_us"]))
    return {
        "x": np.asarray(arrays["x"])[keep].astype(np.int64, copy=False),
        "y": np.asarray(arrays["y"])[keep].astype(np.int64, copy=False),
        "t": timestamps[keep],
        "p": np.asarray(arrays["p"])[keep].astype(np.bool_, copy=False),
    }


def trigger_type_for_edge(edge: str):
    import dv_processing as dv

    edge_text = str(edge).lower()
    if "falling" in edge_text:
        return dv.TriggerType.EXTERNAL_SIGNAL_FALLING_EDGE
    if "pulse" in edge_text:
        return dv.TriggerType.EXTERNAL_SIGNAL_PULSE
    return dv.TriggerType.EXTERNAL_SIGNAL_RISING_EDGE


def trigger_timestamp(record) -> int:
    value = getattr(record, "timestamp", None)
    if value is None and isinstance(record, dict):
        value = record.get("timestamp")
    if callable(value):
        value = value()
    return int(value)


def trigger_edge(record) -> str:
    value = getattr(record, "edge", None)
    if value is None and isinstance(record, dict):
        value = record.get("edge")
    if callable(value):
        value = value()
    return "rising" if value is None else str(value)


def write_trimmed_display_aedat4(
    output_path,
    metadata_path,
    arrays,
    selected_triggers,
    resolution,
    source_aedat4_path,
    buffer_us: int = 2000,
):
    import dv_processing as dv

    selected_triggers = list(selected_triggers)
    if not selected_triggers:
        raise ValueError("selected_triggers must not be empty")
    trigger_ts = np.asarray([trigger_timestamp(trigger) for trigger in selected_triggers], dtype=np.int64)
    window = displayed_trigger_trim_window(trigger_ts, frame_count=len(selected_triggers), buffer_us=int(buffer_us))
    trimmed_arrays = trim_event_arrays_for_time_window(arrays, window)

    output_path = Path(output_path)
    metadata_path = Path(metadata_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        output_path.unlink()

    width, height = (int(resolution[0]), int(resolution[1]))
    config = dv.io.MonoCameraWriter.Config("trimmed_displayed_trigger_window")
    config.addEventStream((width, height), streamName="events", source="trimmed-events")
    config.addTriggerStream(streamName="triggers", source="trimmed-triggers")
    writer = dv.io.MonoCameraWriter(str(output_path), config)

    event_store = dv.EventStore()
    for timestamp, x, y, polarity in zip(
        trimmed_arrays["t"],
        trimmed_arrays["x"],
        trimmed_arrays["y"],
        trimmed_arrays["p"],
        strict=False,
    ):
        event_store.push_back(int(timestamp), int(x), int(y), bool(polarity))
    writer.writeEvents(event_store, streamName="events")

    trigger_packet = dv.TriggerPacket()
    for trigger in selected_triggers:
        trigger_packet.elements.append(dv.Trigger(trigger_timestamp(trigger), trigger_type_for_edge(trigger_edge(trigger))))
    writer.writeTriggerPacket(trigger_packet, streamName="triggers")
    del writer

    metadata = {
        "source_aedat4": str(source_aedat4_path),
        "output_aedat4": str(output_path),
        "selected_trigger_count": int(len(selected_triggers)),
        "buffer_us": int(buffer_us),
        "start_us": int(window["start_us"]),
        "stop_us": int(window["stop_us"]),
        "first_trigger_us": int(window["first_trigger_us"]),
        "last_trigger_us": int(window["last_trigger_us"]),
        "event_count": int(len(trimmed_arrays["t"])),
        "resolution": [width, height],
    }
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
    return metadata


trim_cycles = max(1, ceil(int(TRIMMED_AEDAT4_TRIGGER_COUNT) / max(1, int(cycle_length))))
trim_stages = _process_accumulation_triggers(
    recording.triggers,
    event_arrays["t"],
    window_us=default_window_us,
    window_start_offset_us=default_offset_us,
    max_accumulation_triggers=int(TRIMMED_AEDAT4_TRIGGER_COUNT),
    trigger_cycle_length=cycle_length,
    accumulation_cycles=trim_cycles,
    startup_leader_trigger_count=startup_leader_trigger_count,
)
trim_selected_triggers = list(trim_stages.final[:int(TRIMMED_AEDAT4_TRIGGER_COUNT)])
trim_metadata = write_trimmed_display_aedat4(
    TRIMMED_AEDAT4_PATH,
    TRIMMED_AEDAT4_METADATA_PATH,
    event_arrays,
    trim_selected_triggers,
    (width, height),
    AEDAT4_PATH,
    buffer_us=TRIMMED_AEDAT4_BUFFER_US,
)
print(json.dumps(trim_metadata, indent=2, sort_keys=True))
